We’ll cover:

1️⃣ Connecting to EC2
2️⃣ Installing Node + npm
3️⃣ Setting up the project
4️⃣ Running the app (with `nohup`)
5️⃣ AWS network/security setup (so it’s reachable via IP)

---

## 🚀 STEP-BY-STEP GUIDE

### 1️⃣ Connect to your EC2 instance

From your local terminal:

```bash
ssh -i /path/to/your-key.pem ubuntu@<your-ec2-public-ip>
```

Example:

```bash
ssh -i ~/Downloads/mykey.pem ubuntu@13.232.56.101
```

---

### 2️⃣ Update packages

```bash
sudo apt update && sudo apt upgrade -y
```

---

### 3️⃣ Install Node.js and npm

For Vite projects you need Node ≥ 18 or 20.

```bash
# Option 1: use NodeSource (recommended)
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt install -y nodejs

# verify
node -v
npm -v
```

---

### 4️⃣ Clone or upload your React + Vite project

Either:

#### ✅ Clone from GitHub:

```bash
cd ~
git clone https://github.com/yourusername/yourproject.git
cd yourproject
```

or

#### ✅ Upload (via SCP or FileZilla):

```bash
scp -i /path/to/your-key.pem -r ./yourproject ubuntu@<your-ec2-public-ip>:/home/ubuntu/
```

---

### 5️⃣ Install dependencies

```bash
cd yourproject
npm install
```

---

### 6️⃣ Edit the Vite config (important!)

Open `vite.config.js` and ensure you expose all network interfaces:

```js
export default {
  server: {
    host: '0.0.0.0',
    port: 5173, // default vite port
  },
}
```

---

### 7️⃣ Run the development server (test)

```bash
npm run dev
```

You should see:

```
Local: http://localhost:5173/
Network: http://<your-ec2-private-ip>:5173/
```

✅ If you see that, it’s working locally.

---

### 8️⃣ Allow inbound traffic in AWS

1. Go to **EC2 → Instances → Your Instance → Security → Security groups**
2. Edit **Inbound rules**
3. Add rule:

   * **Type:** Custom TCP
   * **Port range:** 5173
   * **Source:** 0.0.0.0/0 (and optionally ::/0 for IPv6)

Save.

Now anyone can reach `http://<your-ec2-public-ip>:5173`

---

### 9️⃣ Run it in background (nohup)

Stop the current process (`Ctrl +C`) and run:

```bash
nohup npm run dev -- --host 0.0.0.0 > output.log 2>&1 &
```

✅ This will:

* run your Vite dev server in background
* write logs to `output.log`
* survive terminal logout

Check running processes:

```bash
ps aux | grep vite
```

Kill if needed:

```bash
kill -9 <PID>
```

---

### 10️⃣ Access the frontend

Open your browser:

```
http://<your-ec2-public-ip>:5173
```

Example:

```
http://13.232.56.101:5173
```

---

## 🧩 (OPTIONAL) – Build + Serve Static Files

If you prefer to serve a built version (recommended for production):

```bash
npm run build
sudo npm install -g serve
nohup serve -s dist -l 3000 > serve.log 2>&1 &
```

Then update AWS inbound rule:

* Port 3000 → 0.0.0.0/0

Access:

```
http://<your-ec2-public-ip>:3000
```

---

## ✅ Quick Summary

| Step | Command / Action                              | Purpose               |
| ---- | --------------------------------------------- | --------------------- |
| 1    | `ssh -i key.pem ubuntu@ip`                    | connect to EC2        |
| 2    | `sudo apt update && sudo apt upgrade -y`      | update packages       |
| 3    | install Node 20                               | node + npm            |
| 4    | `git clone …` / `scp`                         | get project           |
| 5    | `npm install`                                 | install deps          |
| 6    | edit `vite.config.js` → `host:'0.0.0.0'`      | allow external access |
| 7    | `npm run dev`                                 | test run              |
| 8    | add inbound rule (port 5173/3000 → 0.0.0.0/0) | open port             |
| 9    | `nohup npm run dev -- --host 0.0.0.0 &`       | keep running          |
| 10   | open `http://<EC2-IP>:5173`                   | view app              |

---



# **Nginx reverse-proxy setup** (so your React app runs on port 80 without typing `:5173`)? It’s the usual next step for production hosting.

Perfect 👌 — now let’s continue from your **working Vite/React app** on EC2 and make it run on **port 80 (default HTTP)**, so you can access it just via:
👉 `http://<your-ec2-public-ip>`
without typing `:5173` or `:3000`.

We’ll do this using **Nginx as a reverse proxy**.

---

## 🌐 Full Step-by-Step NGINX Reverse Proxy Setup (Ubuntu EC2)

### 🔹 1️⃣ Install Nginx

```bash
sudo apt update
sudo apt install nginx -y
```

Verify installation:

```bash
sudo systemctl status nginx
```

If not running:

```bash
sudo systemctl start nginx
sudo systemctl enable nginx
```

Now check in browser:
👉 `http://<your-ec2-public-ip>`
You should see the **default Nginx welcome page**.

---

### 🔹 2️⃣ Open Port 80 in AWS Security Group

Go to:

* **EC2 → Instances → Your instance → Security → Security groups**
* Edit **Inbound rules**
  Add:

  * **Type:** HTTP
  * **Port:** 80
  * **Source:** `0.0.0.0/0`
    (and optionally `::/0` for IPv6)

Save.

---

### 🔹 3️⃣ Run your Vite/React app in background (on internal port)

If you already built your app, serve it with:

```bash
npm run build
sudo npm install -g serve
nohup serve -s dist -l 5173 > frontend.log 2>&1 &
```

*(or whatever internal port you want; 5173 is fine)*

Check it’s running:

```bash
curl http://localhost:5173
```

If you see HTML output → ✅ it’s serving fine.

---

### 🔹 4️⃣ Configure Nginx reverse proxy

Now tell Nginx to forward traffic from **port 80 → port 5173**.

Create or edit config file:

```bash
sudo nano /etc/nginx/sites-available/reactapp
```

Add this content:

```nginx
server {
    listen 80;
    server_name _;

    location / {
        proxy_pass http://127.0.0.1:5173;
        proxy_http_version 1.1;
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection 'upgrade';
        proxy_set_header Host $host;
        proxy_cache_bypass $http_upgrade;
    }
}
```

Save (Ctrl + O, Enter, Ctrl + X).

---

### 🔹 5️⃣ Enable the site and reload Nginx

```bash
sudo ln -s /etc/nginx/sites-available/reactapp /etc/nginx/sites-enabled/
sudo nginx -t  # check syntax
sudo systemctl reload nginx
```

---

### 🔹 6️⃣ Test it

Now open your browser:

```
http://<your-ec2-public-ip>
```

✅ You should see your React/Vite app running **without `:5173`**.

---

### 🔹 7️⃣ Optional: Remove default Nginx page

If you want to disable the default page:

```bash
sudo rm /etc/nginx/sites-enabled/default
sudo systemctl reload nginx
```

---

### 🔹 8️⃣ (Optional) Use custom domain with SSL (bonus)

If you later map a domain (e.g., `myapp.com`) to your EC2 IP, you can easily add HTTPS:

```bash
sudo apt install certbot python3-certbot-nginx -y
sudo certbot --nginx -d myapp.com -d www.myapp.com
```

Certbot will automatically generate and configure SSL certificates.

---

## ✅ Summary

| Step | Command / Action                               | Description           |
| ---- | ---------------------------------------------- | --------------------- |
| 1    | `sudo apt install nginx -y`                    | install nginx         |
| 2    | AWS SG inbound rule → port 80                  | open HTTP access      |
| 3    | `nohup serve -s dist -l 5173 &`                | run frontend          |
| 4    | edit `/etc/nginx/sites-available/reactapp`     | proxy config          |
| 5    | `sudo nginx -t && sudo systemctl reload nginx` | apply config          |
| 6    | visit `http://<EC2-IP>`                        | app runs without port |
| 7    | `sudo rm /etc/nginx/sites-enabled/default`     | optional cleanup      |

---


# others

Q : Would you like me to show an alternative production setup (using Node.js + Nginx for SSR or PM2 process management) — so it restarts automatically on reboot? yes give me both setups laso , nodejs+Ngins for ssr pm2

Perfect 👌 — you’re ready to move from basic “frontend-only” hosting to **production-grade deployment** on your EC2.

Below are **two strong setups** you can use, depending on what kind of React app you have:

---

## 🧱 SETUP #1 — Node.js + Nginx (SSR or Express backend)

Use this if your app uses:

* Next.js (Server-Side Rendering), **or**
* React with a custom Node.js/Express backend (API + frontend).

---

### ⚙️ 1️⃣ Install dependencies

```bash
sudo apt update
sudo apt install nginx -y
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
sudo apt install -y nodejs git
node -v && npm -v
```

---

### ⚙️ 2️⃣ Get your project

```bash
cd ~
git clone https://github.com/yourusername/yourproject.git
cd yourproject
npm install
```

If it’s a Next.js app:

```bash
npm run build
npm run start
```

(Default Next.js port is `3000`)

---

### ⚙️ 3️⃣ Configure Nginx Reverse Proxy

Create config:

```bash
sudo nano /etc/nginx/sites-available/nodeapp
```

Paste this:

```nginx
server {
    listen 80;
    server_name _;

    location / {
        proxy_pass http://127.0.0.1:3000;
        proxy_http_version 1.1;
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection 'upgrade';
        proxy_set_header Host $host;
        proxy_cache_bypass $http_upgrade;
    }
}
```

Enable and restart:

```bash
sudo ln -s /etc/nginx/sites-available/nodeapp /etc/nginx/sites-enabled/
sudo nginx -t
sudo systemctl restart nginx
```

✅ Now your Node.js (Next.js/Express) app is accessible at:

```
http://<EC2-public-IP>
```

---

### ⚙️ 4️⃣ Make Node app run continuously (using nohup or PM2 — see below)

If you don’t use PM2 yet:

```bash
nohup npm start > node.log 2>&1 &
```

But for production — **PM2 is best** (see next setup 👇).

---

## 🚀 SETUP #2 — PM2 Process Manager (for Node, Vite, or Next)

PM2 is a **Node process manager** that:

* Keeps your app running 24/7
* Restarts automatically after reboot or crash
* Manages logs easily

---

### ⚙️ 1️⃣ Install PM2 globally

```bash
sudo npm install -g pm2
```

---

### ⚙️ 2️⃣ Start your app with PM2

#### For Vite build (static serve):

```bash
npm run build
sudo npm install -g serve
pm2 start "serve -s dist -l 5173" --name "reactapp"
```

#### For Next.js / Express app:

```bash
pm2 start npm --name "nodeapp" -- start
```

Check status:

```bash
pm2 list
```

Logs:

```bash
pm2 logs
```

Stop/restart:

```bash
pm2 restart reactapp
pm2 stop reactapp
```

---

### ⚙️ 3️⃣ Enable auto-start on reboot

```bash
pm2 startup systemd
```

It will show a command — copy & paste that (example):

```bash
sudo env PATH=$PATH:/usr/bin pm2 startup systemd -u ubuntu --hp /home/ubuntu
```

Then save your process list:

```bash
pm2 save
```

Now, PM2 will:

* Start your app automatically on system boot
* Keep it alive even if you close the SSH session

---

### ⚙️ 4️⃣ Nginx Reverse Proxy (same as before)

If you’re running on 5173 or 3000, configure `/etc/nginx/sites-available/reactapp`:

```nginx
server {
    listen 80;
    server_name _;

    location / {
        proxy_pass http://127.0.0.1:5173;
        proxy_http_version 1.1;
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection 'upgrade';
        proxy_set_header Host $host;
        proxy_cache_bypass $http_upgrade;
    }
}
```

Reload:

```bash
sudo nginx -t
sudo systemctl reload nginx
```

---

## 🧩 Bonus: SSL (HTTPS) with Let’s Encrypt

Once your domain points to your EC2 public IP:

```bash
sudo apt install certbot python3-certbot-nginx -y
sudo certbot --nginx -d yourdomain.com -d www.yourdomain.com
```

It’ll auto-configure HTTPS and renew automatically.

---

## ✅ SUMMARY COMPARISON

| Setup                 | Use Case         | Command to Run App                          | Port | Restart on Boot | Reverse Proxy |
| --------------------- | ---------------- | ------------------------------------------- | ---- | --------------- | ------------- |
| **Vite + Nginx**      | Static frontend  | `nohup serve -s dist -l 5173 &`             | 5173 | ❌               | ✅             |
| **Node/Next + Nginx** | SSR / API        | `npm run start`                             | 3000 | ❌               | ✅             |
| **PM2 + Nginx**       | Production-ready | `pm2 start npm -- start` or `serve -s dist` | any  | ✅               | ✅             |

---



Q : Would you like me to include a **system reboot recovery script** (so your Node + Nginx + PM2 setup auto-starts cleanly after reboot)?
I can give you the exact sequence and test commands for that too.


Perfect 🔥 — this is the **final touch** to make your EC2 frontend (React/Vite/Next.js) deployment **fully production-ready**, even after a reboot or crash.

We’ll configure **auto-start for Node/PM2 + Nginx** and test it.

---

## 🧰 GOAL

After reboot (`sudo reboot` or AWS restart):
✅ Nginx auto-starts
✅ PM2 auto-starts
✅ Node app runs automatically
✅ Your site loads instantly on browser (`http://<your-ec2-ip>`)

---

## ⚙️ STEP-BY-STEP SYSTEM REBOOT AUTO-RECOVERY SETUP

---

### 🔹 1️⃣ Verify Nginx auto-start (enabled by default)

Check:

```bash
sudo systemctl is-enabled nginx
```

If it says `enabled`, good ✅
If not, enable it:

```bash
sudo systemctl enable nginx
```

---

### 🔹 2️⃣ Configure PM2 to start at boot

Run:

```bash
pm2 startup systemd
```

You’ll see output like:

```
[PM2] To setup the Startup Script, copy/paste the following command:
sudo env PATH=$PATH:/usr/bin pm2 startup systemd -u ubuntu --hp /home/ubuntu
```

👉 Copy that entire command and run it exactly.

Example:

```bash
sudo env PATH=$PATH:/usr/bin pm2 startup systemd -u ubuntu --hp /home/ubuntu
```

---

### 🔹 3️⃣ Save your running apps

This ensures PM2 remembers your app list and restarts them automatically.

```bash
pm2 save
```

You should see:

```
[PM2] Saving current process list...
[PM2] Successfully saved
```

---

### 🔹 4️⃣ Test reboot recovery

Reboot your EC2 instance:

```bash
sudo reboot
```

Then reconnect after 1–2 minutes:

```bash
ssh -i /path/to/key.pem ubuntu@<ec2-ip>
```

---

### 🔹 5️⃣ Verify everything restarted

Run:

```bash
pm2 list
```

✅ You should see your apps (like `reactapp` or `nodeapp`) as **online**.

Also check Nginx:

```bash
sudo systemctl status nginx
```

It should show **active (running)**.

---

### 🔹 6️⃣ Access your app in browser

Open:

```
http://<your-ec2-public-ip>
```

If it loads → your recovery setup works perfectly 🎉

---

## 🧩 OPTIONAL — Automatic log rotation (keep logs small)

PM2 logs grow over time; you can auto-rotate them:

```bash
pm2 install pm2-logrotate
pm2 set pm2-logrotate:max_size 10M
pm2 set pm2-logrotate:retain 7
pm2 save
```

This:

* Rotates logs >10 MB
* Keeps last 7 logs
* Runs automatically

---

## ✅ FINAL SUMMARY

| Component    | Purpose                  | Command                                     | Status after reboot |
| ------------ | ------------------------ | ------------------------------------------- | ------------------- |
| **Nginx**    | Reverse proxy (port 80)  | `sudo systemctl enable nginx`               | Auto-start ✅        |
| **PM2**      | Process manager for Node | `pm2 startup`, `pm2 save`                   | Auto-start ✅        |
| **Your App** | React/Next.js            | `pm2 start npm -- start` or `serve -s dist` | Auto-start ✅        |

---

### 🔐 BONUS TIP (optional, but useful)

To auto-restart your Node app if it crashes or gets killed by system OOM:

```bash
pm2 restart all --watch
```

Or run in watch mode permanently:

```bash
pm2 start npm -- start --watch
```

---
